### Chain of thoughts prompting

Tests to see if we can manage some sort of chain of thought prompting. The output might need to be rethought in order to be parsable (pass through another LLM ?).

In [ ]:
# Install packages
# Carefull requires python 3+ (current is 3.13.8)

%pip install langchain
%pip install -U langchain-community
%pip install sentence-transformers
%pip install chromadb

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# install packages 2

#%pip install -r ../statbot-api/requirements.txt
# Warning too long !!

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
  Using cached tiktoken-0.12.0-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (6.7 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached langchain_openai-1.1.1-py3-none-any.whl.metadata (2.6 kB)
INFO: pip 

In [4]:
# Define imports

import pandas as pd
#import os
#import sys
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import PromptTemplate, FewShotPromptTemplate
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
#from langchain.vectorstores.chroma import Chroma
from langchain_community.vectorstores import Chroma
#from langchain.embeddings import HuggingFaceEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

import time
import sys
import os
from sqlalchemyWrapper import (
    schema_db_postgres_statbot_zhaw
)
from langchain_classic.chains import LLMChain
import tiktoken
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts.chat import (
    ChatPromptTemplate,
    HumanMessagePromptTemplate,
    SystemMessagePromptTemplate,
)
from langchain_openai import ChatOpenAI

In [5]:
def few_shot_template_examples(example_prompt, example_selector):
    prefix = '''You are an helpful AI assistant who writes SQL query for a given question. Given the database described by the database schema below, write a SQL query that answers the question.\nDo not explain the SQL query.\nReturn just the query, so it can be run verbatim from your response.\n### Database Schema\n{table_info}
    '''

    few_shot_prompt = FewShotPromptTemplate(
        # These are the examples we want to insert into the prompt.
        example_selector=example_selector,
        example_prompt=example_prompt,
        # The prefix is some text that goes before the examples in the prompt.
        # Usually, this consists of intructions.
        prefix=prefix,
        # The suffix is some text that goes after the examples in the prompt.
        # Usually, this is where the user input will go
        suffix="### Question\n{input}\n### SQL query\n",
        # The input variables are the variables that the overall prompt expects.
        input_variables=["input", "table_info"],
        example_separator="\n\n",
    )
    return few_shot_prompt



def generate_sql_in_context_learning_similar_shots(question, table_name, n_shots=3, file_path="data/query_questions_db.csv"):
    # find the n_shots closest questions from the query_questions_db and the table

    with open(file_path) as f:
        origin_of_shots = pd.read_csv(f, delimiter=',')

    examples = origin_of_shots.loc[origin_of_shots['db_id']==table_name]
    examples = examples.reset_index()
    few_shot_examples = []
    meta_data = []

    for j in range(len(examples)):
        ex_question = examples.loc[j, 'question'].replace("\n", "").strip()
        ex_query = examples.loc[j, 'query']
        few_shot_examples.append({"question": ex_question})
        meta_data.append({"question": ex_question, "query": ex_query})

    to_vectorize = [" ".join(example.values()) for example in few_shot_examples]

    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/distiluse-base-multilingual-cased-v2")

    vectorstore = None
    if vectorstore is not None:
        # CLEAR THE VECTORSTORE
        vectorstore.delete_collection()

    vectorstore = Chroma.from_texts(to_vectorize, embeddings, metadatas=meta_data)

    # Lower score is more similar
    answers = vectorstore.similarity_search_with_score(query=question, k=n_shots)

    examples_selector = SemanticSimilarityExampleSelector(
        vectorstore=vectorstore,
        k=n_shots,
    )
    examples_prompt = PromptTemplate(
        input_variables=["question", "query"],
        template="### Question\n{question}\n### SQL query\n{query}",
    )
    prompt_template = few_shot_template_examples(examples_prompt, examples_selector)

    return prompt_template

In [ ]:
def query_engineering_and_call(question, table_name, qry_id):

    sys.stderr.write(f"inside query_engineering_and_call" + "/n")
    
    prompt_template = generate_sql_in_context_learning_similar_shots(question, table_name)

    # For testing purposes
    # prompt_template = zero_shot_template()

    model_name = os.environ["MODEL_NAME"]
    model_path = os.environ["MODEL_PATH"]

    inference_server_url = os.environ["INFERENCE_SERVER_URL"]
    deployed_llm_token = os.environ["DEPLOYED_LLM_TOKEN"]

    llm = ChatOpenAI(
        model=model_path + model_name,
        openai_api_key=deployed_llm_token,
        openai_api_base=inference_server_url,
        max_tokens=1500,
        n=1,
        stream=False,
        top_p=1.0,
        frequency_penalty=0.0,
        presence_penalty=0.0,
        temperature=0.0
    )

    tic = time.perf_counter()
    
    llm_chain = prompt_template | llm
    sql = None

    ddl = schema_db_postgres_statbot_zhaw(include_tables=['spatial_unit', table_name],
                             sample_number=5)

    llm_inputs = {
        "input": question,
        "table_info": ddl,
    }

    sys.stderr.write(f"llm_inputs: {llm_inputs}\n")

    prompt_strings = prompt_template.format(input = question, table_info = ddl)
    sys.stderr.write(f"Prompt_string: {prompt_strings}\n")

    sys.stderr.write(f"Starting  generation:\n")
    while sql is None:
        try:
            # sql = llm_chain.run(**llm_inputs)
            sql = llm_chain.invoke(llm_inputs)
            sys.stderr.write(f"Question: {question}\n")
            sys.stderr.write(f"sql: {sql}\n")
        except Exception as e:
            sys.stderr.write(str(e))
            time.sleep(3)
            pass
    
    # time 
    toc = time.perf_counter()

    num_tokens = sql.response_metadata['token_usage']['total_tokens']
    sql_response = sql.content
    
    process_time = toc-tic
    print(f"Process Time= {process_time:0.4f} second")
    r = {"message": {
        "db_id": table_name,
        "id": qry_id,
        "generated_query": sql_response.replace("\n", " ").replace("\n\n", " ").replace(" ", " ").replace("  ", " "),
        "prompt": prompt_strings,
        "question": question,
        "time": process_time,
        "num_tokens": num_tokens,
    }}

    return r

### Query check (no database verification)

Ask the LLM to check its query using the original prompt and the generated output. No check with the database (yet).

### Query check (database verification)

Ask the LLM to check its query using the original prompt, the generated output and the database answer. Check can depend on database answer (execution error, empty answer, out of scope answer)